# Embedding-Based Hierarchical Search Demo

This notebook demonstrates the new embedding-based hierarchical search capabilities in CaRL.
We'll show how to:
1. Load and use state embedding models (AE/VAE)
2. Generate subgoals in embedding space
3. Navigate using embedding-conditioned CLLPs
4. Compare with traditional discrete-space approaches

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CaRL imports
from carl.environment.sokoban.env import SokobanEnv
from carl.environment.sokoban.tokenizer import SokobanTokenizer

# Embedding-based components
from carl.components.state_embeddings import StateAutoencoder, StateVAE
from carl.components.embedding_generator import EmbeddingGenerator
from carl.components.embedding_cllp import EmbeddingConditionedCLLP

# Set up environment
tokenizer = SokobanTokenizer(size_of_board=(12, 12))
env = SokobanEnv(tokenizer=tokenizer, num_boxes=4)

print("Environment set up successfully!")

## 1. State Embedding Models

First, let's create and test state embedding models:

In [ ]:
# Model parameters
input_dim = 144  # 12x12 Sokoban board flattened
embedding_dim = 64
batch_size = 8

# Create autoencoder
autoencoder = StateAutoencoder(
    input_dim=input_dim,
    embedding_dim=embedding_dim,
    hidden_dims=[512, 256, 128],
    dropout=0.1
)

# Create VAE
vae = StateVAE(
    input_dim=input_dim,
    embedding_dim=embedding_dim,
    hidden_dims=[512, 256, 128],
    dropout=0.1,
    beta=1.0
)

# Test with random data
test_states = torch.randn(batch_size, input_dim)

# Test autoencoder
ae_output = autoencoder(test_states)
print(f"Autoencoder output keys: {ae_output.keys()}")
print(f"Reconstruction shape: {ae_output['reconstruction'].shape}")
print(f"Embedding shape: {ae_output['embedding'].shape}")

# Test VAE
vae_output = vae(test_states)
print(f"\nVAE output keys: {vae_output.keys()}")
print(f"KL divergence shape: {vae_output['kl_div'].shape}")

# Visualize reconstruction quality
with torch.no_grad():
    reconstruction_error = torch.mean((test_states - ae_output['reconstruction']) ** 2, dim=1)
    print(f"\nMean reconstruction error: {reconstruction_error.mean():.6f}")

## 2. Embedding Generators

Now let's test embedding generators for subgoal prediction:

In [ ]:
# Create embedding generator
generator = EmbeddingGenerator(
    embedding_dim=embedding_dim,
    hidden_dims=[512, 256],
    k_steps=4,
    dropout=0.1,
    use_attention=True
)

# Test subgoal generation
current_embeddings = ae_output['embedding']  # Use embeddings from autoencoder
k_steps = torch.tensor([4, 8, 4, 8, 4, 8, 4, 8])  # Different k values per sample

predicted_subgoals = generator(current_embeddings, k_steps)
print(f"Predicted subgoal embeddings shape: {predicted_subgoals.shape}")

# Compute embedding similarity
cosine_sim = torch.nn.functional.cosine_similarity(
    current_embeddings, predicted_subgoals, dim=1
)
print(f"Cosine similarity between current and predicted embeddings: {cosine_sim.mean():.4f}")

# Visualize embedding norms
current_norms = torch.norm(current_embeddings, dim=1)
predicted_norms = torch.norm(predicted_subgoals, dim=1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(current_norms.detach(), predicted_norms.detach())
plt.xlabel('Current Embedding Norm')
plt.ylabel('Predicted Embedding Norm')
plt.title('Embedding Norm Relationship')
plt.plot([0, 10], [0, 10], 'r--', alpha=0.5)

plt.subplot(1, 2, 2)
plt.hist(cosine_sim.detach().numpy(), bins=10, alpha=0.7)
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.title('Embedding Similarity Distribution')
plt.tight_layout()
plt.show()

## 3. Embedding-Conditioned CLLP

Test the embedding-conditioned conditional low-level policy:

In [ ]:
# Create embedding CLLP
cllp = EmbeddingConditionedCLLP(
    embedding_dim=embedding_dim,
    num_actions=4,  # Sokoban actions: up, down, left, right
    hidden_dims=[512, 256],
    dropout=0.1,
    use_attention=True
)

# Test action prediction
state_embeddings = current_embeddings
subgoal_embeddings = predicted_subgoals

action_logits = cllp(state_embeddings, subgoal_embeddings)
action_probs = cllp.get_action_probs(state_embeddings, subgoal_embeddings)

print(f"Action logits shape: {action_logits.shape}")
print(f"Action probabilities shape: {action_probs.shape}")
print(f"Action probabilities sum to 1: {torch.allclose(action_probs.sum(dim=1), torch.ones(batch_size))}")

# Sample actions
sampled_actions = cllp.sample_action(state_embeddings, subgoal_embeddings, temperature=1.0)
print(f"Sampled actions: {sampled_actions.numpy()}")

# Visualize action distributions
plt.figure(figsize=(12, 3))
for i in range(min(4, batch_size)):
    plt.subplot(1, 4, i+1)
    plt.bar(range(4), action_probs[i].detach().numpy())
    plt.title(f'Sample {i+1}')
    plt.xlabel('Action')
    plt.ylabel('Probability')
    plt.xticks(range(4), ['Up', 'Down', 'Left', 'Right'])
plt.tight_layout()
plt.show()

## 4. End-to-End Pipeline

Demonstrate the complete pipeline from state to action through embeddings:

In [ ]:
def embedding_based_planning_step(state, autoencoder, generator, cllp, k_steps=4):
    """
    Complete embedding-based planning step.
    
    Args:
        state: Input state tensor
        autoencoder: State embedding model
        generator: Subgoal embedding generator 
        cllp: Embedding-conditioned CLLP
        k_steps: Planning horizon
    
    Returns:
        Dictionary with embeddings, subgoals, and actions
    """
    with torch.no_grad():
        # 1. Encode current state
        state_embedding = autoencoder.encode(state)
        
        # 2. Generate subgoal embedding
        k_tensor = torch.tensor([k_steps] * state.shape[0])
        subgoal_embedding = generator(state_embedding, k_tensor)
        
        # 3. Decode subgoal to state space (for visualization)
        subgoal_state = autoencoder.decode(subgoal_embedding)
        
        # 4. Generate action towards subgoal
        action_probs = cllp.get_action_probs(state_embedding, subgoal_embedding)
        action = torch.argmax(action_probs, dim=1)
        
    return {
        'state_embedding': state_embedding,
        'subgoal_embedding': subgoal_embedding,
        'subgoal_state': subgoal_state,
        'action_probs': action_probs,
        'action': action
    }

# Test the complete pipeline
test_state = torch.randn(1, input_dim)  # Single state

result = embedding_based_planning_step(
    test_state, autoencoder, generator, cllp, k_steps=8
)

print("Embedding-based planning step completed:")
print(f"State embedding norm: {torch.norm(result['state_embedding']).item():.4f}")
print(f"Subgoal embedding norm: {torch.norm(result['subgoal_embedding']).item():.4f}")
print(f"Embedding cosine similarity: {torch.nn.functional.cosine_similarity(result['state_embedding'], result['subgoal_embedding']).item():.4f}")
print(f"Selected action: {result['action'].item()} (probability: {result['action_probs'][0, result['action']].item():.4f})")

# Visualize action probabilities
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.bar(range(4), result['action_probs'][0].numpy())
plt.title('Action Probabilities')
plt.xlabel('Action')
plt.ylabel('Probability')
plt.xticks(range(4), ['Up', 'Down', 'Left', 'Right'])

plt.subplot(1, 2, 2)
embedding_diff = result['subgoal_embedding'] - result['state_embedding']
plt.plot(embedding_diff[0].numpy(), alpha=0.7)
plt.title('Embedding Space Movement')
plt.xlabel('Embedding Dimension')
plt.ylabel('Difference')
plt.tight_layout()
plt.show()

## 5. Comparison with Discrete Space

Compare embedding-based approach with traditional discrete-space methods:

In [ ]:
# Simulate discrete vs embedding-based planning
num_planning_steps = 10
state_dim = input_dim
embedding_dim = 64

# Memory and computation comparison
discrete_memory = num_planning_steps * state_dim * 4  # bytes (float32)
embedding_memory = num_planning_steps * embedding_dim * 4  # bytes (float32)

print("Memory Usage Comparison:")
print(f"Discrete space: {discrete_memory:,} bytes ({discrete_memory/1024:.1f} KB)")
print(f"Embedding space: {embedding_memory:,} bytes ({embedding_memory/1024:.1f} KB)")
print(f"Memory reduction: {discrete_memory/embedding_memory:.1f}x")

# Computational complexity comparison
print("\nComputational Complexity:")
print(f"Discrete state operations: O({state_dim}) per step")
print(f"Embedding operations: O({embedding_dim}) per step")
print(f"Complexity reduction: {state_dim/embedding_dim:.1f}x")

# Visualization
categories = ['Memory Usage', 'Computation']
discrete_values = [discrete_memory/1024, state_dim]
embedding_values = [embedding_memory/1024, embedding_dim]

x = np.arange(len(categories))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, discrete_values, width, label='Discrete Space', alpha=0.8)
plt.bar(x + width/2, embedding_values, width, label='Embedding Space', alpha=0.8)

plt.xlabel('Aspect')
plt.ylabel('Size (KB / Dimensions)')
plt.title('Discrete vs Embedding-Based Planning Comparison')
plt.xticks(x, categories)
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## Summary

This demo showed the key components of embedding-based hierarchical search:

1. **State Embeddings**: Compress high-dimensional states into compact representations
2. **Embedding Generators**: Predict subgoal embeddings for hierarchical planning
3. **Embedding CLLPs**: Navigate in embedding space towards subgoals
4. **Efficiency**: Significant reduction in memory and computational requirements

### Key Benefits:
- **Scalability**: Operations in compact embedding space
- **Efficiency**: Reduced memory and computation requirements
- **Generalization**: Embeddings can capture semantic similarities
- **Flexibility**: Works with continuous and discrete environments

### Next Steps:
1. Train models on actual Sokoban data using the provided configs
2. Integrate with CaRL's search algorithms for full hierarchical planning
3. Experiment with different embedding dimensions and architectures
4. Evaluate on more complex environments

For training the models, use the configuration files in `configs/offline_training/sokoban/`.